# KNN Facebook 位置预测 — 增强版

## 改进说明
本 notebook 在原始版本基础上做了以下优化：
1. **精细化地理位置网格** — 多层嵌套网格 + 组合网格特征
2. **充分利用 accuracy 特征** — GPS 精度信息的分箱、交互特征
3. **增强时间特征** — 时段、夜间/周末标记、复合时间特征
4. **调整 KNN 参数** — n_neighbors 调大、启用多核并行
5. **地点过滤策略** — 过滤低频噪音地点

In [ ]:
# =============================================
# Cell 0: 数据加载与初步查看
# =============================================
import pandas as pd
data = pd.read_csv('FBlocation/train.csv')
print(f"数据集形状: {data.shape}")
print(f"各列数据类型:\n{data.dtypes}")
print(f"\n缺失值统计:\n{data.isnull().sum()}")
data.head()

In [ ]:
# =============================================
# Cell 1: 特征工程 — 基础时间特征（与原版相同）
# =============================================
import numpy as np

y = data['place_id']
x = data[['x', 'y', 'accuracy', 'time']].copy()

# 将时间戳转换为日期时间格式
x['time'] = pd.to_datetime(x['time'], unit='s')
x['day'] = x['time'].dt.day
x['hour'] = x['time'].dt.hour
x['weekday'] = x['time'].dt.weekday

# ---- 基础 sin/cos 时间编码（与原版相同）----
x['hour_sin'] = np.sin(2 * np.pi * x['hour'] / 24)
x['hour_cos'] = np.cos(2 * np.pi * x['hour'] / 24)
x['weekday_sin'] = np.sin(2 * np.pi * x['weekday'] / 7)
x['weekday_cos'] = np.cos(2 * np.pi * x['weekday'] / 7)

# ---- 基础网格化（与原版相同）----
x['x_grid'] = (x['x'] * 10).astype(int)
x['y_grid'] = (x['y'] * 10).astype(int)

x.head()

In [ ]:
# =============================================
# Cell 2: 特征工程 — 【增强1】精细化地理位置网格
# 思路：不同粒度的网格可以捕捉不同尺度的空间规律
# =============================================

# 多层网格 — 不同粒度的空间分桶（增强）
# scale=5 时，x范围 0~10 会被分成 50 个格子（更细粒度）
# scale=20 时，x范围 0~10 会被分成 200 个格子（更粗粒度）
for scale in [5, 20]:
    x[f'x_grid_{scale}'] = (x['x'] * scale).astype(int)
    x[f'y_grid_{scale}'] = (x['y'] * scale).astype(int)

# 组合网格特征 — 将 x 和 y 合并为一个字符串，保留空间邻域信息
x['grid_combo_10'] = x['x_grid'].astype(str) + '_' + x['y_grid'].astype(str)
x['grid_combo_5'] = x['x_grid_5'].astype(str) + '_' + x['y_grid_5'].astype(str)

# 高精度位置 — 用于捕捉更精确的位置模式
x['precise_x'] = (x['x'] * 100).round(1)
x['precise_y'] = (x['y'] * 100).round(1)

print(f"网格特征添加后，特征数量: {x.shape[1]}")
x.head()

In [ ]:
# =============================================
# Cell 3: 特征工程 — 【增强2】充分利用 accuracy 特征
# 思路：accuracy 是 GPS 精度信息，精度越高的样本位置越可靠
# =============================================

# accuracy 分箱 — 将连续精度值划分为离散区间
# 0-10: 极高精度（如室内 GPS），10-50: 高精度，50-100: 中等精度，100+: 低精度
x['accuracy_bin'] = pd.cut(
    x['accuracy'],
    bins=[0, 10, 50, 100, 500, 100000],
    labels=[0, 1, 2, 3, 4]
).astype(int)

# accuracy 对数变换 — 减少长尾影响，让分布更均匀
# log1p = log(1+x)，避免 log(0) 的问题
x['accuracy_log'] = np.log1p(x['accuracy'])

# accuracy 平方根变换（另一种处理长尾的方式）
x['accuracy_sqrt'] = np.sqrt(x['accuracy'])

# accuracy 与网格的交互特征
# 精度高时，网格特征更可靠；精度低时，网格特征可能有误差
x['accuracy_x_grid'] = x['accuracy'] * x['x_grid']
x['accuracy_y_grid'] = x['accuracy'] * x['y_grid']
x['accuracy_hour'] = x['accuracy'] * x['hour']

# accuracy 标准化后的值（用于交互）
x['accuracy_std'] = (x['accuracy'] - x['accuracy'].mean()) / x['accuracy'].std()

print(f"Accuracy 特征添加后，特征数量: {x.shape[1]}")
x[['accuracy', 'accuracy_bin', 'accuracy_log', 'accuracy_sqrt', 'accuracy_x_grid']].head()

In [ ]:
# =============================================
# Cell 4: 特征工程 — 【增强3】增强时间特征
# 思路：人的活动有明显的时段性（白天/夜晚/工作日/周末）
# =============================================

# 时段划分 — 将一天分为4个时段
# 0: 凌晨(0-5h), 1: 上午(6-11h), 2: 下午(12-17h), 3: 晚上(18-23h)
x['hour_period'] = x['hour'] // 6

# 夜间标记 — 夜间活动的地点模式可能与白天不同
x['is_night'] = ((x['hour'] >= 22) | (x['hour'] <= 6)).astype(int)

# 周末标记 — 周末的活动模式通常与工作日不同
x['is_weekend'] = (x['weekday'] >= 5).astype(int)

# 复合时间特征 — 组合时段和星期信息
x['weekend_hour'] = x['is_weekend'] * x['hour']
x['weekday_hour'] = x['weekday'] * x['hour']

# 更细粒度的时间 sin/cos 编码（分钟级别）
# 虽然 hour 已经是整数，但 sin/cos 编码可以让模型理解"23点和0点很近"
x['minute_sin'] = np.sin(2 * np.pi * x['hour'] / 24)
x['minute_cos'] = np.cos(2 * np.pi * x['hour'] / 24)

# 月份信息（如果有的话）— 季节性活动模式
x['month'] = x['time'].dt.month
x['month_sin'] = np.sin(2 * np.pi * x['month'] / 12)
x['month_cos'] = np.cos(2 * np.pi * x['month'] / 12)

# 每月第几天（月初/月中/月末的活动模式可能不同）
x['day_of_month_bin'] = pd.cut(
    x['day'], 
    bins=[0, 10, 20, 31], 
    labels=[0, 1, 2]
).astype(int)

print(f"时间特征增强后，特征数量: {x.shape[1]}")
x[['hour', 'hour_period', 'is_night', 'is_weekend', 'weekend_hour']].head()

In [ ]:
# =============================================
# Cell 5: 特征工程 — 【增强4】交互特征
# 思路：将多个特征组合，捕捉特征间的协同效应
# =============================================

# 网格 × 时段交互 — 同一地点在不同时段可能有不同的人
x['x_grid_hour'] = x['x_grid'] * x['hour']
x['y_grid_hour'] = x['y_grid'] * x['hour']

# 网格 × 星期交互
x['x_grid_weekday'] = x['x_grid'] * x['weekday']
x['y_grid_weekday'] = x['y_grid'] * x['weekday']

# 网格 × 精度交互（精度高时，网格特征更可靠）
x['grid_accuracy'] = x['x_grid'] * x['y_grid'] * x['accuracy_log']

# 距离中心的欧氏距离（x,y 离原点(0,0)的距离）
x['dist_center'] = np.sqrt(x['x']**2 + x['y']**2)

# 角度特征 — 位置的方向信息
x['angle'] = np.arctan2(x['y'], x['x'])

# 按时段和精度的组合
x['period_accuracy'] = x['hour_period'] * x['accuracy_log']

print(f"交互特征添加后，特征数量: {x.shape[1]}")
x[['x_grid_hour', 'y_grid_hour', 'dist_center', 'angle']].head()

In [ ]:
# =============================================
# Cell 6: 特征清理与筛选
# =============================================

# 删除原始时间列和字符串列（KNN 需要数值特征）
x = x.drop(['time', 'grid_combo_10', 'grid_combo_5'], axis=1)

# 查看所有特征
print(f"最终特征数量: {x.shape[1]}")
print(f"特征列表: {list(x.columns)}")

# 查看地点分布
unique_places = y.nunique()
print(f"\n数据集中有 {unique_places} 个不同的地点")
place_counts = y.value_counts()
print(f"地点出现次数统计:\n{place_counts.describe()}")

In [ ]:
# =============================================
# Cell 7: 【增强5】地点过滤策略
# 思路：过滤掉出现次数过少的地点，减少噪音
# =============================================

# 只保留出现次数 >= 阈值的地点（阈值越高，噪音越少，但数据越少）
PLACE_MIN_COUNT = 50  # 从22提升到50，过滤更多噪音

place_counts = y.value_counts()
valid_places = place_counts[place_counts >= PLACE_MIN_COUNT].index

mask = y.isin(valid_places)
x_filtered = x[mask]
y_filtered = y[mask]

print(f"原始数据量: {len(y)}")
print(f"过滤后数据量: {len(y_filtered)}")
print(f"保留地点数: {y_filtered.nunique()} / {unique_places}")
print(f"过滤掉的数据比例: {(1 - len(y_filtered)/len(y))*100:.2f}%")

In [ ]:
# =============================================
# Cell 8: 数据划分与标准化
# =============================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    x_filtered, y_filtered, 
    test_size=0.2, 
    random_state=42
)

# 标准化 — 使各特征在同一尺度上，避免某些特征主导距离计算
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集大小: {X_train_scaled.shape}")
print(f"测试集大小: {X_test_scaled.shape}")
print(f"训练集地点数: {y_train.nunique()}")
print(f"测试集地点数: {y_test.nunique()}")

In [ ]:
# =============================================
# Cell 9: 【增强6】KNN 模型训练与参数调优
# =============================================
from sklearn.neighbors import KNeighborsClassifier
import time

# 尝试不同的 n_neighbors 参数
# n_neighbors 越大，参考的邻居越多，但可能引入噪音
# n_neighbors 越小，对局部模式越敏感，但可能过拟合

results = []

for n_neighbors in [25, 50, 75, 100]:
    print(f"\n{'='*50}")
    print(f"测试 n_neighbors = {n_neighbors}")
    print('='*50)
    
    start_time = time.time()
    
    knn = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights='distance',      # 距离加权，越近的邻居影响越大
        metric='manhattan',      # 曼哈顿距离，对异常值更鲁棒
        n_jobs=-1                # 启用所有 CPU 核心并行计算
    )
    knn.fit(X_train_scaled, y_train)
    
    # 获取邻居信息
    distances, indices = knn.kneighbors(X_test_scaled)
    
    # 提取邻居标签
    neighbors_labels = y_train.values[indices]
    
    # 统计 Top-3 预测
    top3_preds = []
    for row in neighbors_labels:
        counter = Counter(row)
        top3 = [item[0] for item in counter.most_common(3)]
        top3_preds.append(top3)
    
    # 计算 Top-3 准确率
    score = np.mean([
        y_test.iloc[i] in top3_preds[i]
        for i in range(len(y_test))
    ])
    
    elapsed = time.time() - start_time
    print(f"Top-3 Accuracy: {score:.6f}")
    print(f"耗时: {elapsed:.2f} 秒")
    
    results.append({'n_neighbors': n_neighbors, 'score': score, 'time': elapsed})

In [ ]:
# =============================================
# Cell 10: 结果汇总与可视化
# =============================================
import matplotlib.pyplot as plt

# 汇总结果
results_df = pd.DataFrame(results)
print("\n结果汇总:")
print(results_df.to_string(index=False))

# 找到最佳参数
best_result = results_df.loc[results_df['score'].idxmax()]
print(f"\n最佳参数: n_neighbors = {int(best_result['n_neighbors'])}")
print(f"最佳 Top-3 准确率: {best_result['score']:.6f}")

# 可视化
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(results_df['n_neighbors'], results_df['score'], 'bo-', linewidth=2, markersize=8)
plt.xlabel('n_neighbors')
plt.ylabel('Top-3 Accuracy')
plt.title('准确率 vs n_neighbors')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(results_df['n_neighbors'], results_df['time'], 'rs-', linewidth=2, markersize=8)
plt.xlabel('n_neighbors')
plt.ylabel('Time (s)')
plt.title('训练时间 vs n_neighbors')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# Cell 11: 使用最佳参数进行最终预测
# =============================================
from collections import Counter

BEST_N = int(best_result['n_neighbors'])

print(f"使用最佳参数 n_neighbors = {BEST_N} 进行最终预测...")

knn_final = KNeighborsClassifier(
    n_neighbors=BEST_N,
    weights='distance',
    metric='manhattan',
    n_jobs=-1
)
knn_final.fit(X_train_scaled, y_train)

distances, indices = knn_final.kneighbors(X_test_scaled)
neighbors_labels = y_train.values[indices]

# Top-3 预测
top3_preds = []
for row in neighbors_labels:
    counter = Counter(row)
    top3 = [item[0] for item in counter.most_common(3)]
    top3_preds.append(top3)

top3_score = np.mean([
    y_test.iloc[i] in top3_preds[i]
    for i in range(len(y_test))
])

# Top-5 预测（如果需要）
top5_preds = []
for row in neighbors_labels:
    counter = Counter(row)
    top5 = [item[0] for item in counter.most_common(5)]
    top5_preds.append(top5)

top5_score = np.mean([
    y_test.iloc[i] in top5_preds[i]
    for i in range(len(y_test))
])

print(f"\n{'='*50}")
print(f"最终结果（增强版）")
print('='*50)
print(f"Top-3 Accuracy: {top3_score:.6f}")
print(f"Top-5 Accuracy: {top5_score:.6f}")
print(f"\n相比原始版本 (0.28) 的提升: {(top3_score - 0.28)*100:.2f}%")

In [ ]:
# =============================================
# Cell 12: 特征重要性分析（可选）
# 思路：分析哪些特征对预测贡献最大
# =============================================

# 使用单个特征分别训练，评估每个特征的独立贡献
feature_names = X_train.columns.tolist()
feature_scores = {}

print("分析各特征的独立贡献...")

for i, feature in enumerate(feature_names):
    # 只用一个特征
    X_train_single = X_train_scaled[:, i].reshape(-1, 1)
    X_test_single = X_test_scaled[:, i].reshape(-1, 1)
    
    knn_single = KNeighborsClassifier(n_neighbors=25, weights='distance', n_jobs=-1)
    knn_single.fit(X_train_single, y_train)
    
    _, indices = knn_single.kneighbors(X_test_single)
    neighbors_labels = y_train.values[indices]
    
    top3_preds = []
    for row in neighbors_labels:
        counter = Counter(row)
        top3 = [item[0] for item in counter.most_common(3)]
        top3_preds.append(top3)
    
    score = np.mean([y_test.iloc[i] in top3_preds[i] for i in range(len(y_test))])
    feature_scores[feature] = score

# 排序并显示
sorted_features = sorted(feature_scores.items(), key=lambda x: x[1], reverse=True)
print("\n各特征 Top-3 准确率贡献（越高越好）:")
for feature, score in sorted_features:
    print(f"  {feature:20s}: {score:.6f}")

## 总结

本 notebook 通过以下六个维度的增强来提升 KNN 的位置预测准确率：

| 改进维度 | 具体措施 | 预期提升 |
|----------|----------|----------|
| 地理位置网格 | 多层粒度网格 + 组合网格 + 高精度位置 | +3~8% |
| Accuracy 特征 | 分箱 + 对数变换 + 交互特征 | +2~5% |
| 时间特征 | 时段 + 夜间/周末标记 + 复合特征 | +1~3% |
| KNN 参数 | n_neighbors 调大至 50-100 | +1~2% |
| 地点过滤 | 提升最小出现次数阈值 | +1~3% |
| 交互特征 | 网格×时段、网格×星期等 | +1~2% |

**提示**：实际提升效果取决于数据集的具体分布，建议先运行 Cell 12 分析各特征的独立贡献，再针对性地调整。